# Signpost-D decoder line (path A): zero-shot baselines, then fine-tuning with seeds

One notebook for every decoder step. Settings are in the first code cell.

- **`MODE = "zeroshot"`** scores each model in `MODELS` untrained (A1): how much size alone gives, and the starting point training must beat.
- **`MODE = "train"`** fine-tunes each model with LoRA for each seed in `SEEDS` (A2–A5) and scores it.
- **Prompt:** the model's own chat template, thinking off. With **`PACKED = True`** (one pass per case) the user turn holds `Case:` + the state once, then every question of the case, numbered, with lettered options (score levels with their positions); the assistant turn is `1: _`, `2: _`, … and each answer is read from the next-token probabilities of the letters ` A`, ` B`, … after `n:` (the `_` placeholders are fixed, so no question sees another's answer). With `PACKED = False`, one pass per question, as A1. No text is generated.
- **Training:** soft-label cross-entropy (+ the expected-level term for score questions), LoRA with α = r on attention, MLP and linear-attention projections, LoRA weights in fp32, fp16 autocast, gradient clipping at 1.0 with the gradient norm logged, warm-up 15%, collapse checks at a step and after epoch 1, best epoch on dev KL. Base weights in fp16 (`"weights": "fp16"`), 4 bit (`"4bit"`) or fp32.
- **Two GPUs, one seed:** with `GPUS = 2` the run is data-parallel: each GPU takes every other micro-batch of cases, the LoRA gradients are averaged across the GPUs before every optimizer step, and dev and benchmark scoring are split between the GPUs. Started from the notebook with `accelerate`'s `notebook_launcher`, so no cell before the last may use CUDA.
- **Data:** v1_3 by default (the encoder's data: three seeds 0.454, their average 0.462 / KL 0.497), pinned dev split (313 cases).
- **Restart:** a checkpoint every `CKPT_EVERY` optimizer steps and after every epoch, and finished runs, in `/kaggle/working/checkpoints` (Kaggle file persistence); "run all" again resumes or skips.
- **Measurement:** dev by type and by question kind; benchmark with case-level 95% bootstrap intervals; every benchmark scoring is counted in `checkpoints/test_touches.json`.

Nothing is uploaded.

In [1]:
# Parameters

MODE = "train"                       # "zeroshot": score untrained models (A1); "train": fine-tune each model for each seed (A2-A5)
MODELS = [                            # name, base weights ("fp16" / "4bit" / "fp32"), LoRA rank, learning rate, cases per GPU per micro-batch, accumulation, eval cases
    {"name": "Qwen/Qwen3-4B", "weights": "fp16", "r": 16, "lr": 1e-4, "batch": 1, "accum": 1, "eval_batch": 4},
]
SEEDS = [42]                          # "train" only; one seed at a time, spread over the GPUs
GPUS = 2                              # data-parallel over this many GPUs (Kaggle "GPU T4 x2"); fewer if fewer are visible
PACKED = True                         # one pass per case: the state once, then all its questions (A1 used one pass per question)
D_SET = "v1_3"                        # the encoder's data (v1_4 adds the twins)
TRAIN_MODE = "all"
LEVEL_POSITIONS = True                # score levels shown as "level i of n (lowest/highest): ..."
EPOCHS = 2                            # best epoch on dev KL (A2: 1 epoch gave 0.5655, loss still falling)
WARMUP_FRAC = 0.15
LORA_DROPOUT = 0.05
SCORE_DIST_W = 0.5
COLLAPSE_STEP, COLLAPSE_MARGIN, ATTEMPTS = 400, 0.02, 2
CKPT_EVERY = 200                      # optimizer steps between mid-epoch checkpoints (a lost session resumes from the last one)
BENCHMARK = True                      # score the test split at the end of each run (counted in checkpoints/test_touches.json)
GROUNDED_TESTS = False                # Signpost's grounded-question tests (slow for big models; not needed for decisions)
RUN_NAME = "11_decoder"

In [2]:
# Cell 1 — Setup: packages, fixed settings, device, progress log

!pip uninstall -y -q torchao      # Kaggle ships torchao 0.10; peft refuses anything below 0.16, and we don't use it
!pip install -q --force-reinstall --no-deps "transformers==5.17.0" "peft==0.21.2"
!pip install -q --no-deps bitsandbytes

import copy, datetime, gc, json, math, os, random, shutil, statistics, subprocess, time

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ["PYTORCH_NVML_BASED_CUDA_CHECK"] = "1"   # GPU checks through NVML: CUDA must not start before the GPU processes are forked
from collections import Counter
from importlib.metadata import version
from pathlib import Path

import pandas as pd
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
import transformers
from huggingface_hub import HfApi, hf_hub_download

HF_TOKEN = os.environ.get("HF_TOKEN") or None     # only for gated models; never printed or written to the notebook
if HF_TOKEN is None:
    try:
        from kaggle_secrets import UserSecretsClient
        for label in ("HF_TOKEN", "hf_token"):
            try:
                HF_TOKEN = UserSecretsClient().get_secret(label) or None
                break
            except Exception:
                pass
    except ImportError:
        pass
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN               # every Hugging Face call (and both GPU processes) uses it; never printed
MAX_PROMPT_TOKENS = 4096                          # a whole case when PACKED
MAX_LETTERS = 26
DEV_FRACTION = 0.15
SEED = 42                                         # the data cell's subsampling seed (unused with TRAIN_MODE "all")
BENCH = "LocalLLaMA/typed-decisions"
LETTERS = [chr(65 + i) for i in range(MAX_LETTERS)]
CKPT = Path("checkpoints")
CKPT.mkdir(exist_ok=True)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")   # each process sets its own GPU in the last cell
RANK, WORLD = 0, 1                                # this process and the number of processes (set again in each process)
AMP = DEVICE.type == "cuda"
RUN_LOG = Path(f"{RUN_NAME}_run.log")
# peft (and bitsandbytes through it) is imported inside each process: importing it here would start CUDA before the launch
assert (transformers.__version__, version("peft")) == ("5.17.0", "0.21.2"), "restart the kernel and run from the top"
api = HfApi()
REV = {BENCH: api.dataset_info(BENCH).sha}


def log(message):
    if RANK:                                      # one process writes the log
        return
    line = f"[{time.strftime('%H:%M:%S')}] {message}"
    print(line, flush=True)
    with open(RUN_LOG, "a") as fh:
        fh.write(line + "\n")


def gpu():
    return f"GPU {torch.cuda.memory_allocated(DEVICE) / 1e9:.1f} GB (peak {torch.cuda.max_memory_allocated(DEVICE) / 1e9:.1f})" if AMP else "CPU"


log(f"{RUN_NAME} | mode {MODE} | models {[m['name'] for m in MODELS]} | seeds {SEEDS if MODE == 'train' else '-'} | data {D_SET} | "
    f"torch {torch.__version__} | GPUs visible {torch.cuda.device_count()} (using up to {GPUS}) | packed {PACKED}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 96.9 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.4 MB/s eta 0:00:00:00:0100:01
[17:57:01] 11_decoder | mode train | models ['Qwen/Qwen3-4B'] | seeds [42] | data v1_3 | torch 2.11.0+cu128 | GPUs visible 2 (using up to 2) | packed True


In [3]:
# Cell 2a — The Signpost-D cases (System One form): one item per question, with the labellers' averaged distribution as the target

DATA_BRANCH = "signpost-typed-decisions"
DATA_DIR = Path("data")
if not (DATA_DIR / "signpost_d").exists():                  # on Kaggle / Colab: clone this branch (an existing clone is refreshed)
    repo = Path("llms-the-hard-way")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", DATA_BRANCH,
                        "https://github.com/skamalj/llms-the-hard-way.git", str(repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", DATA_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo), "checkout", "-B", DATA_BRANCH, "FETCH_HEAD"], check=True)
    DATA_DIR = repo / "data"


def state_text(state):
    return state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)


def render(q):
    """A System One question -> ({candidate name: description}, the gold's keys in the same order). One rendering for our
    cases and the benchmark: noul as true/false with their descriptions, choice by its labels, score by its criteria
    strings, lowest first (keys "0".."n-1")."""
    cr = q.get("criteria") or {}
    if q["type"] == "noul":
        keys = ["true", "false"]
        opts = {k: cr.get(k) or "" for k in keys}
    elif q["type"] == "choice":
        keys = list(cr)
        opts = {k.replace("_", " "): v or "" for k, v in cr.items()}
    else:
        keys = [str(i) for i in range(len(cr))]
        opts = {str(c): "" for c in cr}
    assert len(opts) == len(keys), q
    return opts, keys


d_cases = [json.loads(l) for l in open(DATA_DIR / "signpost_d" / D_SET / "systemone.jsonl")]
SPLIT_SEED = 42                           # the dev split of every run since run 6


def dev_split(ids):
    ids = sorted(ids)
    random.Random(SPLIT_SEED).shuffle(ids)   # fixed: the dev split never depends on the training seed
    return set(ids[: int(len(ids) * DEV_FRACTION)])


ADDED = ("I", "K", "L")                   # cases added after v1 (I: invoices, K: choice, L: score cases): each group split on its own,
TRAIN_ONLY = ("T",)                       # twin cases (v1_4): always training, never dev (a twin and its base must not straddle)
dev_ids = dev_split(c["id"] for c in d_cases if not c["id"].startswith(ADDED + TRAIN_ONLY))   # earlier dev cases stay as before
for g in ADDED:
    dev_ids |= dev_split(c["id"] for c in d_cases if c["id"].startswith(g))


def d_items(c):
    out = []
    for qid, q in c["questions"].items():
        opts, keys = render(q)
        gold = c["gold"][qid]["probabilities"]
        out.append({"task": "d_" + q["type"], "id": c["id"], "qid": qid, "domain": c["meta"]["domain"],
                    "messages": [{"role": "user", "content": state_text(c["state"])}], "current": None,
                    "question": q["instructions"], "options": opts, "levels": list(opts) if q["type"] == "score" else None,
                    "soft": {n: gold[k] for n, k in zip(opts, keys)}})
    return out


train_d = [it for c in d_cases if c["id"] not in dev_ids for it in d_items(c)]
dev_d = [it for c in d_cases if c["id"] in dev_ids for it in d_items(c)]
N_CHOICE_TRAIN = sum(it["task"] == "d_choice" for it in train_d)   # 08b's training set size
if TRAIN_MODE == "choice":
    train_d = [it for it in train_d if it["task"] == "d_choice"]
    dev_d = [it for it in dev_d if it["task"] == "d_choice"]
elif TRAIN_MODE == "all_subsampled":                                 # same count as 08b, all three types
    train_d = random.Random(SEED).sample(train_d, N_CHOICE_TRAIN)
else:
    assert TRAIN_MODE == "all", TRAIN_MODE
print(f"TRAIN_MODE {TRAIN_MODE}: {len(train_d)} training questions (08b had {N_CHOICE_TRAIN}), "
      f"{len({it['id'] for it in train_d})} distinct cases")
print(f"Signpost-D ({D_SET}): {len(d_cases)} cases | train {len(train_d)} questions | dev {len(dev_d)} questions "
      f"({len(dev_ids)} cases) | {dict(Counter(it['task'] for it in train_d))}")

From https://github.com/skamalj/llms-the-hard-way
 * branch            signpost-typed-decisions -> FETCH_HEAD
Reset branch 'signpost-typed-decisions'


Your branch is up to date with 'origin/signpost-typed-decisions'.
TRAIN_MODE all: 8254 training questions (08b had 2846), 1786 distinct cases
Signpost-D (v1_3): 2099 cases | train 8254 questions | dev 1433 questions (313 cases) | {'d_noul': 2359, 'd_choice': 2846, 'd_score': 3049}


In [4]:
# Cell 2b — Grounded-question tests (from Signpost): the general reading check

load = lambda p: json.load(open(DATA_DIR / "modernbert_router" / p))
e12 = load("choice_test/items.json")["items"]
ind_q, ind2_q = load("independent/questions.json")["items"], load("independent2/questions.json")["items"]
print(f"E12 {len(e12)} | independent questions {len(ind_q)} | independent 2 questions {len(ind2_q)} "
      f"(up to {max(len(q['options']) for q in ind2_q)} options)")

E12 170 | independent questions 300 | independent 2 questions 200 (up to 50 options)


In [5]:
# Cell 3 — The decoder: chat-template prompt, answer read from the letter tokens' next-token probabilities

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

SYSTEM = "You read a case and answer one decision question about it. Reply with the letter of the option you choose, and nothing else."
SYSTEM_PACKED = ("You read a case and answer every decision question about it. For each question, write its number and the "
                 "letter of the option you choose.")
LORA_TARGETS = r"^(?!.*(visual|vision)).*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj|in_proj_qkv|in_proj_z|out_proj)$"


def question_block(item):
    """The question and its lettered options (score levels with their positions)."""
    names, task = list(item["options"]), item.get("task", "d_choice")
    if task == "d_noul":
        head = f"Statement: {item['question']}\nIs the statement true or false?"
    elif task == "d_score":
        head = f"Question: {item['question']}\nLevels, lowest first:"
    else:
        head = f"Question: {item['question']}\nOptions:"
    n, lines = len(names), []
    for i, nm in enumerate(names):
        pos = ""
        if task == "d_score" and LEVEL_POSITIONS:
            pos = f"level {i + 1} of {n}" + (" (lowest)" if i == 0 else " (highest)" if i == n - 1 else "") + ": "
        desc = item["options"][nm]
        lines.append(f"{LETTERS[i]}. {pos}{nm}" + (f": {desc}" if desc else ""))
    return head + "\n" + "\n".join(lines)


def by_case(items):
    """Consecutive items of one case (same "id"; items without an id stand alone) -> [[item, ...], ...] in order."""
    groups = []
    for it in items:
        if groups and it.get("id") is not None and groups[-1][0].get("id") == it["id"]:
            groups[-1].append(it)
        else:
            groups.append([it])
    return groups


class SignpostRef:
    """Signpost one-pass exactly as published (our encoder runs' start), through its own loader; zero-shot only."""

    def __init__(self, cfg):
        import importlib.util
        self.cfg = cfg
        self.rev = api.model_info(cfg["name"], token=HF_TOKEN).sha
        spec = importlib.util.spec_from_file_location("signpost_one_pass", DATA_DIR.parent / "signpost" / "signpost_one_pass.py")
        mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
        from huggingface_hub import snapshot_download
        self.sp = mod.SignpostOnePass(snapshot_download(cfg["name"], revision=self.rev, token=HF_TOKEN), device=str(DEVICE))
        log(f"loaded {cfg['name']} @ {self.rev[:8]} (published one-pass encoder, no training) | {gpu()}")

    def train(self, mode=True):
        pass

    def scores(self, items):
        K = max(len(it["options"]) for it in items)
        out = torch.full((len(items), K), -1e4, device=DEVICE)
        for i, it in enumerate(items):
            p = self.sp.probs(it["messages"], it["options"], question=it["question"])
            out[i, : len(p)] = torch.log(torch.tensor([max(p[n], 1e-12) for n in it["options"]], device=DEVICE))
        return out

    def probs(self, item):
        return torch.softmax(self.scores([item]), dim=1)[0].float().cpu()

    def free(self):
        self.sp = None
        gc.collect()
        if AMP:
            torch.cuda.empty_cache()


class DecoderModel:
    """A decoder answering by the next-token probabilities of the letters; LoRA (alpha = r) when training."""

    def __init__(self, cfg, train):
        from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
        self.cfg, name = cfg, cfg["name"]
        self.rev = api.model_info(name, token=HF_TOKEN).sha
        self.tok = AutoTokenizer.from_pretrained(name, revision=self.rev, token=HF_TOKEN)
        self.tok.padding_side = "right"
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        kw = dict(revision=self.rev, token=HF_TOKEN, device_map={"": DEVICE.index or 0} if AMP else None)
        self.weights = cfg.get("weights", "fp32")
        if self.weights == "4bit":
            q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
                                   bnb_4bit_use_double_quant=True)
            m = AutoModelForCausalLM.from_pretrained(name, quantization_config=q, dtype=torch.float16, **kw)
            if train:
                m = prepare_model_for_kbit_training(m, use_gradient_checkpointing=True,
                                                    gradient_checkpointing_kwargs={"use_reentrant": False})
        else:                                                  # fp16: no 4-bit unpacking on every layer (the T4's fast path)
            m = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16 if self.weights == "fp16" else torch.float32, **kw)
            if train:
                m.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        m.config.use_cache = False
        try:
            m.config.get_text_config().use_cache = False
        except Exception:
            pass
        if train:
            m = get_peft_model(m, LoraConfig(r=cfg["r"], lora_alpha=cfg["r"], lora_dropout=LORA_DROPOUT, target_modules=LORA_TARGETS))
            for p in m.parameters():
                if p.requires_grad:
                    p.data = p.data.float()                    # LoRA weights in fp32 (fp16 autocast for the forward pass)
        self.m = m
        base = m.get_base_model() if train else m
        self.inner = base.model.language_model if hasattr(base.model, "language_model") else base.model
        self.head_w = m.get_output_embeddings().weight
        ids = [self.tok((" " if PACKED else "") + L, add_special_tokens=False)["input_ids"] for L in LETTERS]   # " A" after "n:"
        assert all(len(x) == 1 for x in ids), f"a letter is not one token: {ids[:4]}"
        self.label_ids = torch.tensor([x[0] for x in ids], device=DEVICE)
        n_train = sum(p.numel() for p in m.parameters() if p.requires_grad)
        log(f"loaded {name} @ {self.rev[:8]} | weights {self.weights} | trainable {n_train / 1e6:.1f} M | packed {PACKED} | {gpu()}")

    def text(self, item):
        user = "Case:\n" + item["messages"][0]["content"] + "\n\n" + question_block(item)
        msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]
        return self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

    def case_ids(self, case_items):
        """One case in one sequence: the state once, then its questions numbered [1], [2], ...; the assistant turn is
        "1: _", "2: _", ... -> (token ids, for each question the position whose next token is its answer letter)."""
        user = "Case:\n" + case_items[0]["messages"][0]["content"] + "\n\n" + "\n\n".join(
            f"[{i + 1}] " + question_block(it) for i, it in enumerate(case_items))
        msgs = [{"role": "system", "content": SYSTEM_PACKED}, {"role": "user", "content": user}]
        prefix = self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        ids, pos = self.tok(prefix, add_special_tokens=False)["input_ids"], []
        for i in range(len(case_items)):
            ids = ids + self.tok(f"{i + 1}:", add_special_tokens=False)["input_ids"]
            pos.append(len(ids) - 1)
            ids = ids + self.tok(" _\n", add_special_tokens=False)["input_ids"]
        return ids, pos

    def lora_params(self):
        return [p for p in self.m.parameters() if p.requires_grad]

    def train(self, mode=True):
        self.m.train(mode)

    def scores(self, items):
        """-> [items, most options] logits of the option letters at the answer position (missing options -1e4)."""
        if PACKED:
            return self.packed_scores(items)
        enc = self.tok([self.text(it) for it in items], add_special_tokens=False, padding=True, return_tensors="pt")
        ids, mask = enc["input_ids"].to(DEVICE), enc["attention_mask"].to(DEVICE)
        assert ids.shape[1] <= MAX_PROMPT_TOKENS, f"prompt of {ids.shape[1]} tokens"
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
            h = self.inner(input_ids=ids, attention_mask=mask).last_hidden_state
        last = h[torch.arange(len(items), device=DEVICE), mask.sum(1) - 1].float()
        logits = last @ self.head_w[self.label_ids].float().T
        K = max(len(it["options"]) for it in items)
        out = torch.full((len(items), K), -1e4, device=DEVICE)
        for i, it in enumerate(items):
            out[i, : len(it["options"])] = logits[i, : len(it["options"])]
        return out

    def packed_scores(self, items):
        """As scores(), one sequence per case: consecutive items with the same "id" are one case."""
        groups = by_case(items)
        built = [self.case_ids(g) for g in groups]
        L = max(len(x) for x, _ in built)
        assert L <= MAX_PROMPT_TOKENS, f"a case of {L} tokens"
        pad = self.tok.pad_token_id
        ids = torch.tensor([x + [pad] * (L - len(x)) for x, _ in built], device=DEVICE)
        mask = torch.tensor([[1] * len(x) + [0] * (L - len(x)) for x, _ in built], device=DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
            h = self.inner(input_ids=ids, attention_mask=mask).last_hidden_state
        rows = torch.tensor([b for b, (_, pos) in enumerate(built) for _ in pos], device=DEVICE)
        cols = torch.tensor([p for _, pos in built for p in pos], device=DEVICE)
        logits = h[rows, cols].float() @ self.head_w[self.label_ids].float().T
        flat = [it for g in groups for it in g]
        K = max(len(it["options"]) for it in flat)
        out = torch.full((len(flat), K), -1e4, device=DEVICE)
        for i, it in enumerate(flat):
            out[i, : len(it["options"])] = logits[i, : len(it["options"])]
        return out

    def probs(self, item):
        with torch.no_grad():
            return torch.softmax(self.scores([item]), dim=1)[0].float().cpu()

    def free(self):
        self.m = self.inner = self.head_w = None
        gc.collect()
        if AMP:
            torch.cuda.empty_cache()

In [6]:
# Cell 4a — The loss: soft labels (+ an expected-level term for score questions); metrics on the dev cases
# (accuracy against the label's top option, KL, Brier, ECE)

def soft_loss(scores, items):
    """Cross-entropy against the labellers' distribution (= KL + a constant). Score questions add
    SCORE_DIST_W x |expected level - target expected level| / (levels - 1), each option weighted by its level in the
    rubric (it["levels"]), not by its place after the options are shuffled."""
    K = scores.shape[1]
    target = torch.tensor([[it["soft"][n] for n in it["options"]] + [0.0] * (K - len(it["options"])) for it in items],
                          device=scores.device)
    target = target / target.sum(1, keepdim=True)
    logp = F.log_softmax(scores, dim=1)
    loss = -(target * logp).sum(1).mean()
    is_score = torch.tensor([it["task"] == "d_score" for it in items], device=scores.device)
    if is_score.any():
        lvl = torch.tensor([[it["levels"].index(n) if it["levels"] else 0 for n in it["options"]] + [0] * (K - len(it["options"]))
                            for it in items], device=scores.device, dtype=scores.dtype)
        n_levels = torch.tensor([len(it["options"]) for it in items], device=scores.device, dtype=scores.dtype)
        gap = ((logp.exp() * lvl).sum(1) - (target * lvl).sum(1)).abs() / (n_levels - 1).clamp(min=1)
        loss = loss + SCORE_DIST_W * gap[is_score].mean()
    return loss


def barrier():
    if WORLD > 1:
        dist.barrier()


def spread(jobs, fn):
    """fn(list of jobs) -> one result per job. The jobs are shared out over the GPUs (every WORLD-th to each) and the
    results gathered back in the original order on every GPU."""
    mine = list(range(RANK, len(jobs), WORLD))
    res = fn([jobs[i] for i in mine])
    if WORLD == 1:
        return res
    parts = [None] * WORLD
    dist.all_gather_object(parts, list(zip(mine, res)))
    out = [None] * len(jobs)
    for part in parts:
        for i, r in part:
            out[i] = r
    return out


def d_predictions(model, items, batch=4):
    """-> [(item, {option: probability})] for Signpost-D items, in order; batch = cases per forward pass."""
    model.train(False)
    groups = by_case(items)

    def run(gs):
        out = []
        with torch.no_grad():
            for s0 in range(0, len(gs), batch):
                chunk = gs[s0:s0 + batch]
                rows = iter(torch.softmax(model.scores([it for g in chunk for it in g]), dim=1).float().cpu().tolist())
                for g in chunk:
                    out.append([{n: row[j] for j, n in enumerate(it["options"])} for it, row in zip(g, rows)])
        return out

    return [(it, p) for g, ps in zip(groups, spread(groups, run)) for it, p in zip(g, ps)]


def soften(p, temperature):
    """Probabilities softened as p ** (1 / T), renormalised."""
    q = [max(v, 1e-12) ** (1 / temperature) for v in p]
    z = sum(q)
    return [v / z for v in q]


def t_of(temperature, task):
    """A temperature: one number, or one per question type ({"d_noul": T, ...})."""
    return temperature.get(task, 1.0) if isinstance(temperature, dict) else temperature


def d_metrics(preds, temperature=1.0):
    """The benchmark's four measures, against our own soft labels: accuracy (top option = the label's top option), KL from the
    label, Brier, ECE (10 bins). With a temperature (one, or one per question type), probabilities are softened."""
    rows = []
    for it, p in preds:
        q = {n: max(v, 1e-12) ** (1 / t_of(temperature, it["task"])) for n, v in p.items()}
        z = sum(q.values())
        q = {n: v / z for n, v in q.items()}
        g = it["soft"]
        ok = max(q, key=q.get) == max(g, key=g.get)
        rows.append({"task": it["task"], "ok": ok, "conf": max(q.values()),
                     "kl": sum(gv * math.log(gv / max(q[n], 1e-9)) for n, gv in g.items() if gv > 0),
                     "brier": sum((q[n] - g[n]) ** 2 for n in g)})
    def summ(rs):
        n, bins = len(rs), [[0, 0, 0.0] for _ in range(10)]
        for r in rs:
            b = bins[min(int(r["conf"] * 10), 9)]; b[0] += 1; b[1] += r["ok"]; b[2] += r["conf"]
        ece = sum(b[0] / n * abs(b[1] / b[0] - b[2] / b[0]) for b in bins if b[0])
        return {"n": n, "accuracy": round(sum(r["ok"] for r in rs) / n, 4), "kl": round(statistics.mean(r["kl"] for r in rs), 4),
                "brier": round(statistics.mean(r["brier"] for r in rs), 4), "ece": round(ece, 4)}
    return {"all": summ(rows), **{t: summ([r for r in rows if r["task"] == t]) for t in sorted({r["task"] for r in rows})}}


def fit_temperatures(preds):
    """One temperature per question type, each minimising KL on our own dev cases of that type (never the benchmark)."""
    grid = [0.5, 0.75, 1, 1.25, 1.5, 2, 2.5, 3, 4, 5]
    out = {}
    for task in sorted({it["task"] for it, _ in preds}):
        sub = [(it, p) for it, p in preds if it["task"] == task]
        out[task] = min(grid, key=lambda t: d_metrics(sub, t)["all"]["kl"])
    return out


def calibration(pairs):
    """pairs: [(predicted probabilities, gold probabilities)], same option order. Do the predictions sit nearer 0 and 1 than
    the gold? Top-probability histograms side by side, mean normalised entropy, and the over-certain options: predicted
    under 0.01 where the gold has 0.05 or more, with the share of the total KL they cause (scorer's floor 1e-9)."""
    edges = [0, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 1.0001]
    hist = lambda tops: [sum(lo <= t < hi for t in tops) for lo, hi in zip(edges, edges[1:])]
    ent = lambda v: -sum(x * math.log(x) for x in v if x > 0) / math.log(len(v)) if len(v) > 1 else 0.0
    n_opt = over = 0
    kl_all = kl_over = 0.0
    for p, g in pairs:
        for pi, gi in zip(p, g):
            n_opt += 1
            term = gi * math.log(gi / max(pi, 1e-9)) if gi > 0 else 0.0
            kl_all += term
            if pi < 0.01 and gi >= 0.05:
                over += 1
                kl_over += term
    return {"top_p_bins": [f"{lo:.2f}-{min(hi, 1):.2f}" for lo, hi in zip(edges, edges[1:])],
            "pred_top_hist": hist([max(p) for p, _ in pairs]), "gold_top_hist": hist([max(g) for _, g in pairs]),
            "pred_entropy": round(statistics.mean(ent(p) for p, _ in pairs), 4),
            "gold_entropy": round(statistics.mean(ent(g) for _, g in pairs), 4),
            "over_certain_options": over, "of_options": n_opt,
            "kl_share_from_over_certain": round(kl_over / kl_all, 4) if kl_all else 0.0}

In [7]:
# Cell 4b — Training: data-parallel over the GPUs (LoRA gradients averaged before each step), gradient accumulation,
# clipping with the norm logged, collapse checks, best epoch on dev KL, checkpoints every CKPT_EVERY steps and after
# every epoch (resumed after a lost session)

class Collapsed(Exception):
    """The predictions stayed at a uniform guess."""


def uniform_kl(items):
    return statistics.mean(sum(g * math.log(g * len(it["soft"])) for g in it["soft"].values() if g > 0) for it in items)


def train_model(model, seed, ckdir):
    cfg = model.cfg
    B, ACC = cfg["batch"], cfg["accum"]
    train_cases = by_case(train_d)

    def micro_batches(epoch):
        """Micro-batches of B cases in one order on every GPU (seed + epoch); micro-batch k goes to GPU k % WORLD. Cut to a
        whole number of optimizer steps."""
        cs = train_cases[:]
        random.Random(seed + epoch).shuffle(cs)
        mbs = [cs[i:i + B] for i in range(0, len(cs), B)]
        return mbs[: len(mbs) // (WORLD * ACC) * WORLD * ACC]

    def prepare(case, epoch):
        """Choice and true/false options shuffled (score levels keep their order) and, packed, the questions too: with their
        own seed per case and epoch, so a resumed run repeats them exactly."""
        rng = random.Random(f"{seed}-{epoch}-{case[0]['id']}")
        case = [dict(it) for it in case]
        for it in case:
            if it["task"] != "d_score":
                names = list(it["options"]); rng.shuffle(names); it["options"] = {n: it["options"][n] for n in names}
        if PACKED:
            rng.shuffle(case)
        return case

    per_epoch = len(micro_batches(1)) // (WORLD * ACC)
    total = EPOCHS * per_epoch
    warm = max(1, int(WARMUP_FRAC * total))
    params = model.lora_params()
    if WORLD > 1:                                              # the same start on every GPU
        for p in params:
            dist.broadcast(p.data, 0)
    torch.manual_seed(seed * 100 + RANK)                       # dropout differs between GPUs
    opt = torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=0.01)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, max(0.0, (total - s) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    collapse_at = min(COLLAPSE_STEP, max(50, per_epoch // 2))
    best, step, history, run, norms, start_epoch = (float("inf"), None, 0), 0, [], [], [], 1
    latest = ckdir / "latest.pt"
    if latest.exists():                                        # resume from the last checkpoint (mid-epoch or after an epoch)
        st = torch.load(latest, map_location="cpu", weights_only=False)
        by_name = dict(model.m.named_parameters())
        for n, v in st["lora"].items():
            by_name[n].data.copy_(v)
        opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"]); scaler.load_state_dict(st["scaler"])
        step, history, run, best = st["step"], st["history"], st["run"], (st["best_kl"], st["best_lora"], st["best_epoch"])
        start_epoch = st["epoch"] + 1 if st["complete"] else st["epoch"]
        log(f"RESUMED at step {step} (epoch {st['epoch']}{' complete' if st['complete'] else ''}) from {latest}")
    log(f"training {cfg['name']} seed {seed} on {WORLD} GPU(s): {per_epoch} optimizer steps per epoch "
        f"({B} case(s) x {ACC} x {WORLD} GPUs per step), {total} steps")
    t0, start_step = time.perf_counter(), step

    def save(epoch, complete):
        if RANK == 0:
            ckdir.mkdir(parents=True, exist_ok=True)
            torch.save({"epoch": epoch, "complete": complete, "step": step, "history": history, "run": run[-500:],
                        "lora": {n: p.detach().cpu() for n, p in model.m.named_parameters() if p.requires_grad},
                        "opt": opt.state_dict(), "sched": sched.state_dict(), "scaler": scaler.state_dict(),
                        "best_kl": best[0], "best_lora": best[1], "best_epoch": best[2]}, ckdir / "latest.pt.tmp")
            os.replace(ckdir / "latest.pt.tmp", latest)
            log(f"  checkpoint saved at step {step} (epoch {epoch}{' complete' if complete else ''})")
        barrier()

    def optimizer_step(epoch):
        nonlocal step
        if WORLD > 1:                                          # average the (still scaled) gradients over the GPUs, so
            grads = [p.grad if p.grad is not None else torch.zeros_like(p) for p in params]   # every GPU steps alike
            flat = torch._utils._flatten_dense_tensors(grads)
            dist.all_reduce(flat)
            flat /= WORLD
            for p, g in zip(params, torch._utils._unflatten_dense_tensors(flat, grads)):
                p.grad = g
        scaler.unscale_(opt)
        norms.append(float(torch.nn.utils.clip_grad_norm_(params, 1.0)))
        scaler.step(opt); scaler.update(); sched.step()
        opt.zero_grad(set_to_none=True)
        step += 1
        if step % 100 == 0 or step == total:
            el = time.perf_counter() - t0
            gn = [x for x in norms[-100:] if math.isfinite(x)]
            log(f"  epoch {epoch} step {step}/{total} | loss {statistics.mean(run[-100 * ACC:]):.4f} | grad norm mean "
                f"{statistics.mean(gn) if gn else float('nan'):.2f} max {max(gn) if gn else float('nan'):.2f} | "
                f"~{(total - step) / max(1e-9, (step - start_step) / el) / 60:.1f} min left | {gpu()}")
        if step == collapse_at:
            check = dev_d[:300]
            kl, base = d_metrics(d_predictions(model, check, batch=cfg["eval_batch"]))["all"]["kl"], uniform_kl(check)
            log(f"  collapse check at step {step}: dev KL {kl:.4f} vs uniform {base:.4f}")
            if kl > base - COLLAPSE_MARGIN:
                raise Collapsed(f"dev KL {kl:.4f} at step {step}")
            model.train(True)
        if step % CKPT_EVERY == 0 and step % per_epoch:
            save(epoch, False)

    for epoch in range(start_epoch, EPOCHS + 1):
        model.train(True)
        opt.zero_grad(set_to_none=True)
        mbs = micro_batches(epoch)
        for s in range(step - (epoch - 1) * per_epoch, per_epoch):   # steps of this epoch not done yet
            for a in range(ACC):
                items = [it for case in mbs[(s * ACC + a) * WORLD + RANK] for it in prepare(case, epoch)]
                loss = soft_loss(model.scores(items), items)
                scaler.scale(loss / ACC).backward()
                run.append(loss.item())
            optimizer_step(epoch)
        dev = d_metrics(d_predictions(model, dev_d, batch=cfg["eval_batch"]))["all"]
        history.append({"epoch": epoch, **dev})
        if epoch == 1 and dev["kl"] > uniform_kl(dev_d) - COLLAPSE_MARGIN:
            raise Collapsed(f"dev KL {dev['kl']:.4f} after epoch 1")
        log(f"epoch {epoch} done | dev {dev}")
        if dev["kl"] < best[0]:
            best = (dev["kl"], {n: p.detach().cpu().clone() for n, p in model.m.named_parameters() if p.requires_grad}, epoch)
        save(epoch, True)
    by_name = dict(model.m.named_parameters())
    for n, v in best[1].items():
        by_name[n].data.copy_(v.to(by_name[n].device))
    minutes = (time.perf_counter() - t0) / 60
    log(f"training done in {minutes:.1f} min | best epoch {best[2]} (dev KL {best[0]:.4f})")
    return {"train_minutes": minutes, "best_epoch": best[2], "history": history, "gpus": WORLD}

In [8]:
# Cell 5a — Timing and the question test (from Signpost's 03)

def lat_summary(ms):
    ms = sorted(ms)
    return {"p50": ms[len(ms) // 2], "p95": ms[min(len(ms) - 1, int(0.95 * len(ms)))], "n": len(ms)}


def timed_probs(model, item):
    if AMP:
        torch.cuda.synchronize(DEVICE)
    t0 = time.perf_counter()
    p = model.probs(item)
    if AMP:
        torch.cuda.synchronize(DEVICE)
    return p, (time.perf_counter() - t0) * 1000


def questions(model, items):
    rows, lat = [], []
    for it in items:
        p, ms = timed_probs(model, {"messages": [{"role": "user", "content": it["state"]}], "current": None,
                                    "question": it["question"], "options": {o: "" for o in it["options"]},
                                    "task": {"noul": "d_noul", "score": "d_score"}.get(it["type"], "d_choice")})
        top = int(p.argmax())
        rows.append({"type": it["type"], "ok": it["options"][top] == it["answer"],
                     "level_error": abs(top - it["answer_index"]) if it["type"] == "score" else None})
        lat.append(ms)
    return rows, lat

In [9]:
# Cell 5b — The grounded-question tests on a model: R = {test: (right, total)}, LAT = {test: latency}

TYPE = {"noul": "yes/no", "choice": "choice", "score": "score", "many_choice": "many-option"}


def evaluate(model):
    model.train(False)
    R, LAT = {}, {}
    for name, items in [("E12", e12), ("IND", ind_q), ("IND2", ind2_q)]:
        rows, LAT[name] = questions(model, items)
        R[f"{name} questions"] = (sum(r["ok"] for r in rows), len(rows))
        for t in TYPE:
            rs = [r for r in rows if r["type"] == t]
            if rs:
                R[f"{name} {TYPE[t]}"] = (sum(r["ok"] for r in rs), len(rs))
        log(f"{name} questions {R[f'{name} questions'][0]}/{len(rows)} | " +
            " · ".join(f"{TYPE[t]} {R[f'{name} {TYPE[t]}'][0]}/{R[f'{name} {TYPE[t]}'][1]}" for t in TYPE if f"{name} {TYPE[t]}" in R))
    return R, LAT

In [10]:
# Cell 5c — The benchmark, scored once at the end (general mode: never used for training or for any choice above).
# Scoring copied from the benchmark's bench/score_with_jobs.py (score_one, 10-bin ECE).

def bench_split(split):
    f = hf_hub_download(BENCH, f"all/{split}-00000-of-00001.parquet", repo_type="dataset", revision=REV[BENCH])
    rows = pd.read_parquet(f).to_dict("records")
    for r in rows:
        r["state"], r["questions"], r["gold"] = (json.loads(r[k]) for k in ("state", "questions", "gold"))
    return rows


def score_one(pred, gold):
    opts = list(gold["probabilities"].keys())
    g = [float(gold["probabilities"][o]) for o in opts]
    if gold["type"] == "noul":
        pred = dict(pred)
        pred["probabilities"] = {{"yes": "true", "no": "false"}.get(k, k): v for k, v in pred["probabilities"].items()}
    p = [float(pred["probabilities"].get(o, 0.0)) for o in opts]
    s = sum(p)
    p = [v / s for v in p] if s > 0 else [1.0 / len(opts)] * len(opts)
    am_p, am_g = p.index(max(p)), g.index(max(g))
    kl = sum(gi * math.log(gi / max(pi, 1e-9)) for gi, pi in zip(g, p) if gi > 0)
    brier = sum((pi - gi) ** 2 for pi, gi in zip(p, g))
    return am_p == am_g, kl, brier, max(p)


def bench_item(case, q):
    """A benchmark question as a Signpost item, through the same render() as our own cases (state as its JSON text, as
    /v1/systemone sends it)."""
    opts, keys = render(q)
    return {"messages": [{"role": "user", "content": state_text(case["state"])}], "current": None,
            "question": q.get("instructions") or "", "options": opts}, keys


def bench_score(model, cases, temperature=1.0, pairs=None):
    """The four measures on the benchmark. temperature: one number, or one per question type. pairs (a list), if given,
    collects (prediction, gold) in gold order for calibration()."""
    model.train(False)
    rows = []
    for case in cases:
        for qid, q in case["questions"].items():
            item, keys = bench_item(case, q)
            p = soften(model.probs(item).tolist(), t_of(temperature, "d_" + q["type"]))
            pred = {"probabilities": dict(zip(keys, p))}
            ok, kl, brier, conf = score_one(pred, case["gold"][qid])
            rows.append({"type": q["type"], "workflow": case["workflow"], "ok": ok, "kl": kl, "brier": brier, "conf": conf})
            if pairs is not None:
                g = case["gold"][qid]["probabilities"]
                pairs.append(([pred["probabilities"].get(k, 0.0) for k in g], [float(v) for v in g.values()]))
    def summ(rs):
        n, bins = len(rs), [[0, 0, 0.0] for _ in range(10)]
        for r in rs:
            b = bins[min(int(r["conf"] * 10), 9)]; b[0] += 1; b[1] += r["ok"]; b[2] += r["conf"]
        ece = sum(b[0] / n * abs(b[1] / b[0] - b[2] / b[0]) for b in bins if b[0])
        return {"n": n, "accuracy": round(sum(r["ok"] for r in rs) / n, 4), "kl_from_gold": round(statistics.mean(r["kl"] for r in rs), 4),
                "brier": round(statistics.mean(r["brier"] for r in rs), 4), "ece": round(ece, 4)}
    out = {"all": summ(rows)}
    for f in ("type", "workflow"):
        out.update({f"{f}={v}": summ([r for r in rows if r[f] == v]) for v in sorted({r[f] for r in rows})})
    return out

In [11]:
# Cell 5d — Benchmark items carry their type, so the prompt asks a statement, a choice or a scale as in training

_bench_item = bench_item


def bench_item(case, q):
    item, keys = _bench_item(case, q)
    item["task"] = "d_" + q["type"]
    return item, keys

In [12]:
# Cell 6 — Run every model (and seed): train if MODE is "train", then dev by type and kind, benchmark with case-level
# 95% intervals (each scoring counted), results and dev predictions saved; finished runs are skipped on a rerun.
# With GPUS > 1 every process runs this same loop on its own GPU (accelerate's notebook_launcher); process 0 writes.

KIND = {(c["id"], q): c["meta"]["kinds"].get(q, "?") for c in d_cases for q in c["questions"]}
TOUCHES = CKPT / "test_touches.json"


def count_test_touch(label):
    if RANK:
        return
    t = json.load(open(TOUCHES)) if TOUCHES.exists() else []
    t.append({"time": time.strftime("%Y-%m-%d %H:%M"), "run": label})
    json.dump(t, open(TOUCHES, "w"), indent=1)
    log(f"benchmark test split scored: touch number {len(t)} ({label})")


def summ_rows(rs):
    n, bins = len(rs), [[0, 0, 0.0] for _ in range(10)]
    for r in rs:
        b = bins[min(int(r["conf"] * 10), 9)]; b[0] += 1; b[1] += r["ok"]; b[2] += r["conf"]
    ece = sum(b[0] / n * abs(b[1] / b[0] - b[2] / b[0]) for b in bins if b[0])
    return {"n": n, "accuracy": round(sum(r["ok"] for r in rs) / n, 4), "kl_from_gold": round(statistics.mean(r["kl"] for r in rs), 4),
            "brier": round(statistics.mean(r["brier"] for r in rs), 4), "ece": round(ece, 4)}


def bench_probs(model, cases):
    """Per case, each question's probabilities in render() key order (one forward pass per case, the cases shared out
    over the GPUs)."""
    model.train(False)

    def run(jobs):
        out = []
        with torch.no_grad():
            for ci, case in jobs:
                built = [bench_item(case, q) for q in case["questions"].values()]
                items = [dict(item, id=f"bench-{ci}") for item, _ in built]
                P = torch.softmax(model.scores(items), dim=1).float().cpu()
                out.append([P[j, : len(keys)].tolist() for j, (_, keys) in enumerate(built)])
        return out

    return spread(list(enumerate(cases)), run)


def bench_eval(cases, probs, T):
    """The four measures as the official scorer, plus 95% bootstrap intervals over cases for accuracy and KL."""
    rows = []
    for ci, (case, ps) in enumerate(zip(cases, probs)):
        for (qid, q), p in zip(case["questions"].items(), ps):
            _, keys = render(q)
            pl = soften(p, t_of(T, "d_" + q["type"]))
            ok, kl, brier, conf = score_one({"probabilities": dict(zip(keys, pl))}, case["gold"][qid])
            rows.append({"case": ci, "type": q["type"], "workflow": case["workflow"], "ok": ok, "kl": kl, "brier": brier, "conf": conf})
    out = {"all": summ_rows(rows)}
    for f in ("type", "workflow"):
        out.update({f"{f}={v}": summ_rows([r for r in rows if r[f] == v]) for v in sorted({r[f] for r in rows})})
    by_case_rows = {}
    for r in rows:
        by_case_rows.setdefault(r["case"], []).append(r)
    groups, rng, accs, kls = list(by_case_rows.values()), random.Random(0), [], []
    for _ in range(1000):
        sample = [r for g in (rng.choice(groups) for _ in groups) for r in g]
        accs.append(sum(r["ok"] for r in sample) / len(sample)); kls.append(statistics.mean(r["kl"] for r in sample))
    accs.sort(); kls.sort()
    out["ci95"] = {"accuracy": [round(accs[25], 4), round(accs[974], 4)], "kl_from_gold": [round(kls[25], 4), round(kls[974], 4)]}
    return out


def by_kind(preds, T):
    kinds = sorted({KIND.get((it["id"], it["qid"]), "?") for it, _ in preds})
    return {k: d_metrics([(it, p) for it, p in preds if KIND.get((it["id"], it["qid"]), "?") == k], T)["all"] for k in kinds}


def run_dir(cfg, seed):
    short = cfg["name"].split("/")[-1]
    return CKPT / f"dec-{short}-{D_SET}-{MODE}-pos{int(LEVEL_POSITIONS)}" / (
        f"zeroshot-{'packed' if PACKED else 'single'}" if seed is None else
        f"lr{cfg['lr']:g}-r{cfg['r']}-ep{EPOCHS}-{cfg.get('weights', 'fp32')}-{'packed' if PACKED else 'single'}-seed{seed}")


def run_all():
    for cfg in MODELS:
        short = cfg["name"].split("/")[-1]
        if cfg.get("kind") == "signpost" and MODE != "zeroshot":
            continue                                           # the reference is scored zero-shot only
        for seed in ([None] if MODE == "zeroshot" else SEEDS):
            tag = f"{short} {MODE}" + (f" seed {seed}" if seed is not None else "")
            rdir = run_dir(cfg, seed)
            if (rdir / "done.json").exists():
                log(f"===== {tag}: finished earlier, read back from {rdir}")
                continue
            log(f"===== {tag} ===== | {WORLD} GPU(s) | {gpu()}")
            trained, attempts, model = None, [], None
            for attempt in range(1 if seed is None else ATTEMPTS):
                s = (seed or 0) + 1000 * attempt
                random.seed(s); torch.manual_seed(s)           # the same LoRA start on every GPU
                model = SignpostRef(cfg) if cfg.get("kind") == "signpost" else DecoderModel(cfg, train=seed is not None)
                if seed is None:
                    break
                try:
                    trained = train_model(model, s, rdir / f"attempt-seed{s}")
                    attempts.append({"seed": s, "status": "ok"})
                    break
                except Collapsed as e:
                    log(f"attempt {attempt + 1} COLLAPSED: {e}")
                    attempts.append({"seed": s, "status": f"collapsed: {e}"})
                    model.free(); model = None
            else:
                if seed is not None:
                    log(f"{tag}: collapsed in every attempt; skipped")
                    continue
            dev_preds = d_predictions(model, dev_d, batch=cfg["eval_batch"])
            T = fit_temperatures(dev_preds)
            res = {"model": cfg["name"], "revision": model.rev, "mode": MODE, "seed": seed, "attempts": attempts, "d_set": D_SET,
                   "level_positions": LEVEL_POSITIONS, "packed": PACKED, "gpus": WORLD, "config": cfg, "trained": trained,
                   "temperature": T, "dev": {"T=1": d_metrics(dev_preds), "T per type": d_metrics(dev_preds, T)},
                   "dev_by_kind": by_kind(dev_preds, T)}
            log(f"{tag} dev (T per type): " + " | ".join(f"{k} {v}" for k, v in res["dev"]["T per type"].items()))
            log(f"{tag} dev by kind: " + " | ".join(f"{k} acc {v['accuracy']} KL {v['kl']}" for k, v in res["dev_by_kind"].items()))
            if GROUNDED_TESTS and RANK == 0:
                R, _ = evaluate(model)
                res["question_tests"] = {k: list(v) for k, v in R.items()}
            barrier()
            if BENCHMARK:
                count_test_touch(tag)
                P = bench_probs(model, TEST)
                res["benchmark"] = {"T=1": bench_eval(TEST, P, 1.0), "T per type": bench_eval(TEST, P, T)}
                b = res["benchmark"]["T per type"]
                log(f"{tag} benchmark (T per type): {b['all']} | 95% CI acc {b['ci95']['accuracy']} KL {b['ci95']['kl_from_gold']}")
                for g, m_ in b.items():
                    if g not in ("all", "ci95") and RANK == 0:
                        print(f"  {g:38s} acc {m_['accuracy']:.3f}  KL {m_['kl_from_gold']:.3f}  Brier {m_['brier']:.3f}  "
                              f"ECE {m_['ece']:.3f}  (n {m_['n']})")
            if RANK == 0:
                rdir.mkdir(parents=True, exist_ok=True)
                json.dump(res, open(rdir / "results.json", "w"), indent=1)
                dump = [{"id": it["id"], "qid": it["qid"], "t": it["task"][2:], "p": [round(p[n], 4) for n in it["options"]]}
                        for it, p in dev_preds]
                json.dump(dump, open(rdir / "dev_preds.json", "w"))
                print(f"DEVPRED_JSON {tag.replace(' ', '_')} " + json.dumps(dump, separators=(",", ":")))
                if seed is not None:
                    model.m.save_pretrained(rdir / "adapter")
                    for a in rdir.glob("attempt-seed*/latest.pt"):
                        a.unlink()
                done = {"dev": res["dev"]["T per type"]["all"], "dev_by_kind": res["dev_by_kind"],
                        "bench": res.get("benchmark", {}).get("T per type", {}).get("all"),
                        "bench_ci95": res.get("benchmark", {}).get("T per type", {}).get("ci95"),
                        "bench_types": {k: v for k, v in res.get("benchmark", {}).get("T per type", {}).items() if k.startswith("type=")}}
                json.dump(done, open(rdir / "done.json", "w"), indent=1)
            barrier()
            model.free(); model = None


def worker():
    """One process: its GPU from the launcher's environment, then the whole loop."""
    global RANK, WORLD, DEVICE
    WORLD, RANK = int(os.environ.get("WORLD_SIZE", "1")), int(os.environ.get("RANK", "0"))
    if WORLD > 1:
        DEVICE = torch.device(f"cuda:{int(os.environ['LOCAL_RANK'])}")
        torch.cuda.set_device(DEVICE)
        dist.init_process_group("nccl", rank=RANK, world_size=WORLD, timeout=datetime.timedelta(hours=1))
    try:
        run_all()
    finally:
        if WORLD > 1:
            dist.destroy_process_group()


TEST = bench_split("test") if BENCHMARK else None
N_GPU = min(GPUS, torch.cuda.device_count()) if AMP else 1
log(f"starting on {N_GPU} GPU(s)")
if N_GPU > 1:
    assert not torch.cuda.is_initialized(), "CUDA was started in this kernel before the launch: restart the kernel and run all"
    from accelerate import notebook_launcher
    notebook_launcher(worker, num_processes=N_GPU)
else:
    worker()

RANK, WORLD = 0, 1
SUMMARY = {}
for cfg in MODELS:
    for seed in ([None] if MODE == "zeroshot" else SEEDS):
        f = run_dir(cfg, seed) / "done.json"
        if f.exists() and not (cfg.get("kind") == "signpost" and MODE != "zeroshot"):
            SUMMARY[f"{cfg['name'].split('/')[-1]} {MODE}" + (f" seed {seed}" if seed is not None else "")] = json.load(open(f))
print("\nreferences (benchmark, T per type): encoder three stable seeds 0.454 / KL 0.527, their average 0.462 / KL 0.497; "
      "run 9 decoder (Qwen3-0.6B, v1_1) 0.461 / 0.491; untrained Qwen3-4B 0.499 / 0.557; prior 0.470 / 0.347; "
      "d3-lite 0.8B 0.531, d3-nano 2B 0.629, d3-mini 4B 0.743")
for tag, v in SUMMARY.items():
    b, ci = v.get("bench") or {}, v.get("bench_ci95") or {}
    log(f"{tag}: dev acc {v['dev']['accuracy']} KL {v['dev']['kl']} | bench acc {b.get('accuracy')} {ci.get('accuracy')} "
        f"KL {b.get('kl_from_gold')} {ci.get('kl_from_gold')} Brier {b.get('brier')} ECE {b.get('ece')} | "
        + " ".join(f"{k[5:]} {t['accuracy']}" for k, t in v.get("bench_types", {}).items()))
json.dump(SUMMARY, open(f"{RUN_NAME}_{MODE}_summary.json", "w"), indent=1)

all/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  222kB            

all/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

[17:57:14] starting on 2 GPU(s)
Launching training on 2 CUDAs.
[17:57:15] ===== Qwen3-4B train seed 42 ===== | 2 GPU(s) | GPU 0.0 GB (peak 0.0)


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

...

[17:58:15] loaded Qwen/Qwen3-4B @ 1cfa9a72 | weights fp16 | trainable 33.0 M | packed True | GPU 8.2 GB (peak 8.2)
[17:58:15] training Qwen/Qwen3-4B seed 42 on 2 GPU(s): 893 optimizer steps per epoch (1 case(s) x 1 x 2 GPUs per step), 893 steps


/tmp/ipykernel_114/219189145.py:87: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step()
/tmp/ipykernel_114/219189145.py:87: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step()


[18:02:21]   epoch 1 step 100/893 | loss 1.2801 | grad norm mean 5.75 max 21.51 | ~32.4 min left | GPU 8.7 GB (peak 9.3)
[18:06:35]   epoch 1 step 200/893 | loss 0.8593 | grad norm mean 3.17 max 7.02 | ~28.8 min left | GPU 8.7 GB (peak 9.4)
[18:06:35]   checkpoint saved at step 200 (epoch 1)


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


[18:10:47]   epoch 1 step 300/893 | loss 0.7788 | grad norm mean 3.14 max 8.40 | ~24.8 min left | GPU 8.7 GB (peak 9.4)
[18:14:57]   epoch 1 step 400/893 | loss 0.8138 | grad norm mean 2.76 max 6.60 | ~20.6 min left | GPU 8.7 GB (peak 9.4)
[18:15:13]   collapse check at step 400: dev KL 0.2894 vs uniform 0.6641
[18:15:14]   checkpoint saved at step 400 (epoch 1)
[18:19:22]   epoch 1 step 500/893 | loss 0.7573 | grad norm mean 2.89 max 13.19 | ~16.6 min left | GPU 8.7 GB (peak 9.4)
[18:23:32]   epoch 1 step 600/893 | loss 0.8362 | grad norm mean 2.71 max 15.79 | ~12.3 min left | GPU 8.7 GB (peak 9.4)
[18:23:33]   checkpoint saved at step 600 (epoch 1)
[18:27:39]   epoch 1 step 700/893 | loss 0.7394 | grad norm mean 2.51 max 5.27 | ~8.1 min left | GPU 8.7 GB (peak 9.5)
[18:31:50]   epoch 1 step 800/893 | loss 0.7928 | grad norm mean 2.62 max 6.04 | ~3.9 min left | GPU 8.7 GB (peak 9.5)
[18:31:51]   checkpoint saved at step 800 (epoch 1)
[18:35:44]   epoch 1 step 893/893 | loss 0.6966 | g